`Filter()` removes data from your RDD.
 - Takes a function that returns a boolean



In [3]:
from pyspark import SparkConf, SparkContext

In [4]:
conf = SparkConf().setMaster("local").setAppName("FilteringTemperature")
sc = SparkContext(conf=conf)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/01 15:40:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/05/01 15:40:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/05/01 15:40:44 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [1]:

def parseLine(line):
    fields = line.split(',')
    stationID = fields[0]
    entryType = fields[2]
    temperature = float(fields[3]) * 0.1 * (9.0/5.0) + 32
    return (stationID, entryType, temperature)

In [5]:
lines = sc.textFile("/opt/spark/data/1800.csv")
parsedLines = lines.map(parseLine)

In [6]:
# Filter out all buyt tmin entries

minTemps = parsedLines.filter(lambda x: "TMIN" in x[1])

In [7]:
# create (stationID, temperature) key/val paris
stationTemps = minTemps.map(lambda x: (x[0], x[2]))

In [8]:
#min temperature for the entire year for every station
minTemps = stationTemps.reduceByKey(lambda x,y: min(x,y))

In [11]:
# collect and print the results

results = minTemps.collect()

for result in results:
    print(result[0] + f"\t {result[1]}")

ITE00100554	 5.359999999999999
EZE00100082	 7.699999999999999


In [12]:
maxTemps = parsedLines.filter(lambda x: "TMAX" in x[1])
stationTempsMax = maxTemps.map(lambda x: (x[0], x[2]))
maxTemps = stationTempsMax.reduceByKey(lambda x,y: max(x,y))
results_max = maxTemps.collect()
for result in results_max:
    print(result[0] + f"\t {result[1]}")

ITE00100554	 90.14000000000001
EZE00100082	 90.14000000000001
